# Chapter 26: Sliding Window

The programs from Chapter 26 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## A Fixed-Size Window

`window.py`

In [1]:
"""Sliding window: a range [left, right] that moves through a sequence.
The right edge adds one item at a time; the left edge drops items when
the window breaks a rule. Each item enters and leaves once: O(n)."""

from collections import Counter, deque


def max_sum_k(nums, k):
    """The largest sum of k neighboring numbers (a fixed-size window)."""
    window = sum(nums[:k])
    best = window
    for right in range(k, len(nums)):
        window += nums[right] - nums[right - k]  # slide by one
        best = max(best, window)
    return best


def longest_unique(text):
    """Length of the longest substring with no repeated character."""
    last_seen, left, best = {}, 0, 0
    for right, ch in enumerate(text):
        if last_seen.get(ch, -1) >= left:
            left = last_seen[ch] + 1  # jump past the earlier copy
        last_seen[ch] = right
        best = max(best, right - left + 1)
    return best


def shortest_with_sum(nums, target):
    """Length of the shortest run of positive numbers with sum >= target,
    or 0. Grow on the right; shrink from the left while still enough."""
    left, total, best = 0, 0, 0
    for right, x in enumerate(nums):
        total += x
        while total >= target:
            if not best or right - left + 1 < best:
                best = right - left + 1
            total -= nums[left]
            left += 1
    return best


def min_window(text, needed):
    """The shortest substring of text containing every letter of needed
    (with repeats), or ""."""
    need, missing = Counter(needed), len(needed)
    left, best = 0, ""
    for right, ch in enumerate(text):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        while missing == 0:  # the window has everything: shrink it
            if not best or right - left + 1 < len(best):
                best = text[left : right + 1]
            need[text[left]] += 1
            if need[text[left]] > 0:
                missing += 1
            left += 1
    return best


def window_maxima(nums, k):
    """The maximum of every window of size k. A deque keeps positions of
    possible maxima with their values decreasing from front to back."""
    q, out = deque(), []
    for right, x in enumerate(nums):
        while q and nums[q[-1]] <= x:
            q.pop()  # smaller and older: can never be a maximum again
        q.append(right)
        if q[0] <= right - k:
            q.popleft()  # fell out of the window
        if right >= k - 1:
            out.append(nums[q[0]])
    return out

## Demos and Tests

`window_demo.py`

In [2]:
"""The sliding-window solutions on small examples."""

from window import (
    longest_unique,
    max_sum_k,
    min_window,
    shortest_with_sum,
    window_maxima,
)

print(max_sum_k([2, 1, 5, 1, 3, 2], 3))
print(longest_unique("abcabcbb"), longest_unique("pwwkew"))
print(shortest_with_sum([2, 3, 1, 2, 4, 3], 7))
print(repr(min_window("ADOBECODEBANC", "ABC")))
print(window_maxima([1, 3, -1, -3, 5, 3, 6, 7], 3))

9
3 3
2
'BANC'
[3, 3, 5, 5, 6, 7]


## Timing

`bench_window.py`

In [3]:
"""Maximum of every window of size k = n / 10: recomputing each window
against the monotonic deque."""

import random

import bench
from window import window_maxima

rng = random.Random(2026)


def brute(nums, k):
    return [max(nums[i : i + k]) for i in range(len(nums) - k + 1)]


def make(n):
    return [rng.random() for _ in range(n)], n // 10


sizes = [1_000, 2_000, 4_000, 8_000]
results = {
    "max of each window": bench.doubling(brute, make, sizes),
    "monotonic deque": bench.doubling(window_maxima, make, sizes),
}
bench.table(results)
bench.chart(results, "ch26_window.png", "Window maxima, k = n / 10")

         n  max of each window  monotonic deque
     1,000             1.08 ms           158 us
     2,000         4.2 ms x3.9      323 us x2.0
     4,000        16.2 ms x3.8      662 us x2.1
     8,000        64.2 ms x4.0     1.37 ms x2.1
